In [ ]:
import csv
from pathlib import Path
from orders_utils import normalize_city, parse_amount
import json


DATA_DIR = Path.cwd().parent / 'data'
REQUIRED = {'order_id', 'city', 'amount'}


def audit_orders(rows: list[dict]) -> tuple[list[dict], list[dict]]:
    """Берет список словарей. Проверяет наличие проблем через check_row. Если проблемы есть, добавляет их в список словарей
    problems. Если проблем нет, кладет очищенные строки в clean_rows. Возвращает кортеж из 2х списков словарей."""
    problems = []
    clean_rows = []    
    for row in rows:
        check = check_row(row)
        if check:
            problems.append(check)                
        else:
            clean_row = {'order_id': row['order_id'], 
                         'city': normalize_city(row['city']), 
                         'amount': parse_amount(row['amount'])}
            clean_rows.append(clean_row)     
    return clean_rows, problems


def check_row(row: dict) -> list[dict]:
    """Проверяет в словаре по ключу 'city'- если None или значение '', добавляет 
    словарь с id, названием поля текущим значением и причиной.
     Проверяет значение amount. Если None, добавляет словарь с id, полем, текущим значением и причиной
     Принимает словарь, возвращает список словарей(может быть 0,1,2 проблем)"""
    problems = []
    if row['city'] is None or row['city'].strip() =='':
        city_problem = {'order_id': row['order_id'], 'field': 'city', 'raw_value': row['city'], 'reason': 'пусто'}
        problems.append(city_problem)
    if parse_amount(row['amount']) is None:
        amount_problem = {'order_id': row['order_id'], 'field': 'amount','raw_value': row['amount'], 'reason': 'не число'}
        problems.append(amount_problem)    
    return problems


def read_orders(path: Path) -> list[dict]:
    """Читает CSV с заказами. Падает с ValueError, если нет обязательных столбцов."""
    with open(path, encoding='utf-8', newline='') as f:
        reader = csv.DictReader(f)
        missing = REQUIRED - set(reader.fieldnames)   # fieldnames — список названий столбцов
        if missing:
            raise ValueError(f'в файле {path.name} нет столбцов: {missing}')
        return list(reader)


def write_clean(clean_rows: list[dict], path: Path) -> None:    
    """записывает чистые строки через DictWriter: столбцы order_id, city, amount, заголовок, все строки."""
    with open(path, 'w', encoding='utf-8', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=['order_id', 'city', 'amount'])
        writer.writeheader()
        writer.writerows(clean_rows)
    print("Файл записан.")

    
def run_audit(raw_path: Path, clean_path: Path, report_path: Path) -> None:
    """Читает сырой файл, проводит аудит, пишет чистый CSV и отчёт JSON."""
    try:
        rows = read_orders(raw_path)
    except ValueError as e:
        print(f'плохой файл: {e}')
    except FileNotFoundError as e:
        print(f'файл не найден: {e.filename}')
    else:
        clean, probs = audit_orders(rows)
        write_clean(clean, clean_path)        
        with open(report_path, 'w', encoding='utf-8') as f:
            json.dump(probs, f, ensure_ascii=False, indent=2)
        print(f'чистых строк: {len(clean)}, проблем: {len(probs)}')

    
run_audit(DATA_DIR / 'orders_raw.csv',       
          DATA_DIR / 'audit_clean.csv',      
          DATA_DIR / 'audit_problems.json')  

run_audit(DATA_DIR / 'nope.csv',
          DATA_DIR / 'audit_clean.csv',
          DATA_DIR / 'audit_problems.json')

run_audit(DATA_DIR / 'orders_bad.csv',
          DATA_DIR / 'audit_clean.csv',
          DATA_DIR / 'audit_problems.json')

    


Файл записан.
чистых строк: 3 ,проблем: 2
файл не найден: c:\Projects\ml-journey\data\nope.csv
плохой файл: в файле orders_bad.csv нет столбцов: {'order_id'}
